# Multi-Agent Financial Analysis System
**AAI-520 Final Project | University of San Diego**
Andre Hoth & Ken Lai

---

## Project Overview

This notebook demonstrates an autonomous Investment Research Agent that, given a
stock ticker symbol, plans its own research strategy, retrieves financial and news
data from multiple sources, routes that content to specialized analyst agents,
synthesizes a coherent investment analysis, evaluates and refines its own output,
and retains memory across runs to improve future analyses.

### Rubric Requirements Demonstrated

| Requirement | Component | Module |
|---|---|---|
| Agent Function: Planning | InvestmentResearchAgent | agents/investment_agent.py |
| Agent Function: Dynamic Tool Use | Tools Layer | tools/ |
| Agent Function: Self-Reflection | InvestmentResearchAgent | agents/investment_agent.py |
| Agent Function: Memory | AgentMemory | memory/agent_memory.py |
| Workflow 1: Prompt Chaining | NewsProcessingChain | workflows/prompt_chain.py |
| Workflow 2: Routing | ContentRouter | workflows/router.py |
| Workflow 3: Evaluator-Optimizer | EvaluatorOptimizer | workflows/evaluator_optimizer.py |

### GitHub Repository
[aai-520-multi-agent-financial](https://github.com/andrehoth/aai-520-multi-agent-financial)

---

In [ ]:
# ── Imports and Configuration ─────────────────────────────────────────────────
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Tools
from tools.price_data import fetch_price_history, fetch_price_summary
from tools.news_data import fetch_news
from tools.fundamentals import fetch_earnings, fetch_income_statement
from tools.macro_data import fetch_macro_indicators
from tools.llm import call_llm, LLM_PROVIDER, LLM_MODEL

# Agents
from agents.schema import empty_analysis, ANALYSIS_SCHEMA, ContentType
from agents.earnings_agent import EarningsAnalyzer
from agents.news_agent import NewsAnalyzer
from agents.market_agent import MarketAnalyzer
from agents.investment_agent import InvestmentResearchAgent

# Workflows
from workflows.prompt_chain import NewsProcessingChain
from workflows.router import ContentRouter

# Memory
from memory.agent_memory import save_memory, load_memory, clear_memory

# Demonstration ticker
TICKER = "AAPL"

print(f"Ticker:       {TICKER}")
print(f"LLM Provider: {LLM_PROVIDER}")
print(f"LLM Model:    {LLM_MODEL}")
print(f"Timestamp:    {datetime.now().isoformat()}")

---

## 2. Design Rationale: Assignment Requirements to Architecture

The following mapping drives all architectural decisions in this notebook.
All data sources are drawn from the assignment brief's recommended list.

### Rubric Requirements to Specialist Agents to Data Sources

The routing workflow requires three specialist agents. The agent responsibilities
determine the data domains, which in turn determine the tool files.

| Rubric Requirement | Specialist Agent | Data Domain | Tool | Source |
|---|---|---|---|---|
| Routing: Earnings | EarningsAnalyzer | Fundamentals | fundamentals.py | Alpha Vantage |
| Routing: News | NewsAnalyzer | Financial news | news_data.py | NewsAPI |
| Routing: Market | MarketAnalyzer | Price data | price_data.py | yfinance |
| Routing: Market | MarketAnalyzer | Macro context | macro_data.py | FRED API |

### Agent Function: Dynamic Tool Use

The assignment requires the agent to "use tools dynamically (APIs, datasets,
retrieval)." Each tool is called only when the agent's research plan calls for
it. The instructor confirmed that an explicit RAG implementation is not required;
the four API tool calls satisfy the retrieval requirement in scope
(Mokhtari Jadid, K., personal communication, September 2026).

### Inter-Agent Schema Decision

A narrative dict is used to pass analysis between components rather than a typed
structured schema. Each specialist agent produces a narrative string. Synthesis
weaves those strings into a final analysis. The evaluator scores the synthesis
narrative using an LLM rubric prompt. See docs/architecture.md Section 6.1 for
the full design decision and tradeoffs.

---

## 3. Inter-Agent Data Schema

The shared analysis dict is defined in `agents/schema.py`. All pipeline
components read from and write to this dict. Fields are initialized to empty
values upfront so that any stage can safely reference any key without checking
existence first, and a falsy check reliably indicates whether a stage has run.

In [ ]:
# ── Inter-Agent Data Schema ───────────────────────────────────────────────────
# Demonstrates the shared analysis dict that flows through the entire pipeline.
# Fields are populated progressively as each pipeline stage completes.
# See agents/schema.py for full field definitions and design decision rationale.

analysis = empty_analysis(TICKER)

print(f"Analysis dict initialized for {TICKER}:")
for key, value in analysis.items():
    print(f"  {key}: {repr(value)}")

print(f"\nContentType values available to router: {[ct.value for ct in ContentType]}")

---

## 4. Agent Function: Dynamic Tool Use

The four data tools are called dynamically based on the agent's research plan.
Each tool wraps a specific API and returns structured data for its domain.
Real API calls are made at runtime and no mock data is used in this notebook.

### Tool 1: Price Data (yfinance)

Fetches OHLCV price history and summary statistics for the demonstration ticker.
Serves the MarketAnalyzer specialist agent.

In [ ]:
# ── Tool 1: Price Data ────────────────────────────────────────────────────────
# yfinance wrapper; no API key required (open source library, no registration)
# Downstream consumer: MarketAnalyzer (agents/market_agent.py)

price_summary = fetch_price_summary(TICKER)
price_history = fetch_price_history(TICKER, period="3mo")

print(f"Price Summary for {TICKER}:")
for key, value in price_summary.items():
    print(f"  {key}: {value}")

print(f"\nPrice History (last 5 trading days):")
print(price_history.tail())

In [ ]:
# ── Visualization: Price History ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(price_history.index, price_history["Close"], color="steelblue", linewidth=1.5)
ax1.axhline(
    y=price_summary["week_52_high"],
    color="green",
    linestyle="--",
    alpha=0.7,
    label="52-week high",
)
ax1.axhline(
    y=price_summary["week_52_low"],
    color="red",
    linestyle="--",
    alpha=0.7,
    label="52-week low",
)
ax1.set_ylabel("Price (USD)")
ax1.set_title(f"{TICKER} Price History (3 months)")
ax1.legend()

ax2.bar(
    price_history.index, price_history["Volume"] / 1e6, color="steelblue", alpha=0.6
)
ax2.axhline(
    y=price_summary["avg_volume"] / 1e6,
    color="orange",
    linestyle="--",
    alpha=0.7,
    label="Avg volume",
)
ax2.set_ylabel("Volume (millions)")
ax2.set_xlabel("Date")
ax2.legend()

plt.tight_layout()
plt.show()

Price history and volume data fed to MarketAnalyzer for trend and momentum analysis.

### Tool 2: Financial News (NewsAPI)

Fetches recent English-language financial news articles for the demonstration
ticker. Serves the NewsProcessingChain and NewsAnalyzer specialist agent.
Language filtering via langdetect and query refinement are applied at fetch time.

In [ ]:
# ── Tool 2: Financial News ────────────────────────────────────────────────────
# NewsAPI wrapper; API key required (free tier, register at newsapi.org)
# Downstream consumer: NewsProcessingChain -> NewsAnalyzer (agents/news_agent.py)
# Language filtering and query refinement applied at fetch time

articles = fetch_news(TICKER)

print(f"News articles fetched for {TICKER}: {len(articles)}")
print()
for i, article in enumerate(articles[:3], 1):
    print(f"Article {i}:")
    print(f"  Title:  {article['title']}")
    print(f"  Source: {article['source']}")
    print(f"  Date:   {article['published_at']}")
    print()

### Tool 3: Earnings and Fundamentals (Alpha Vantage)

Fetches quarterly earnings history and income statement data for the
demonstration ticker. Serves the EarningsAnalyzer specialist agent.
Results are limited to the 8 most recent quarters for relevance.

In [ ]:
# ── Tool 3: Earnings and Fundamentals ────────────────────────────────────────
# Alpha Vantage wrapper; API key required (free tier, register at alphavantage.co)
# Downstream consumer: EarningsAnalyzer (agents/earnings_agent.py)
# Limited to 8 most recent quarters

earnings = fetch_earnings(TICKER)
income = fetch_income_statement(TICKER)

if earnings:
    print(f"Earnings history for {TICKER} (most recent 3 quarters):")
    for q in earnings[:3]:
        print(
            f"  {q['fiscalDateEnding']}: EPS {q['reportedEPS']} vs estimate "
            f"{q['estimatedEPS']} (surprise: {q['surprisePercentage']}%)"
        )
else:
    print("Earnings data unavailable (Alpha Vantage rate limit reached)")

if earnings:
    print(f"Earnings history for {TICKER} (most recent 3 quarters):")
    for q in earnings[:3]:
        print(
            f"  {q['fiscalDateEnding']}: EPS {q['reportedEPS']} vs estimate "
            f"{q['estimatedEPS']} (surprise: {q['surprisePercentage']}%)"
        )
else:
    print(
        "Earnings data returned empty. Possible causes: API rate limit, "
        "network error, or invalid ticker. Check Alpha Vantage quota at "
        "alphavantage.co/support."
    )

### Tool 4: Macroeconomic Indicators (FRED API)

Fetches macroeconomic indicators including the federal funds rate, CPI, and
unemployment rate. Serves the MarketAnalyzer specialist agent, providing macro
context that differentiates it from the EarningsAnalyzer.

In [ ]:
# ── Tool 4: Macroeconomic Indicators ─────────────────────────────────────────
# FRED API wrapper; API key required (free, register at fred.stlouisfed.org)
# Downstream consumer: MarketAnalyzer (agents/market_agent.py)
# Provides macro context distinct from company-level earnings data

macro = fetch_macro_indicators()

print("Macroeconomic indicators (most recent observation):")
for indicator, observations in macro.items():
    latest = observations[-1]
    print(f"  {indicator}: {latest['value']} ({latest['date']})")

In [ ]:
# ── Visualization: Macroeconomic Indicators ───────────────────────────────────
# Shows fed funds rate, CPI, and unemployment rate over the analysis period.
# Three indicators are plotted separately due to different scales and units.
# Feeds MarketAnalyzer for macroeconomic context analysis.

fed_dates = [
    datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["fed_funds_rate"]
]
fed_values = [obs["value"] for obs in macro["fed_funds_rate"]]

cpi_dates = [datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["cpi"]]
cpi_values = [obs["value"] for obs in macro["cpi"]]

unemp_dates = [
    datetime.strptime(obs["date"], "%Y-%m-%d") for obs in macro["unemployment_rate"]
]
unemp_values = [obs["value"] for obs in macro["unemployment_rate"]]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax1.plot(fed_dates, fed_values, color="steelblue", linewidth=1.5)
ax1.set_ylabel("Fed Funds Rate (%)")
ax1.set_title("Macroeconomic Indicators (2024 to present)")

ax2.plot(cpi_dates, cpi_values, color="darkorange", linewidth=1.5)
ax2.set_ylabel("CPI")

ax3.plot(unemp_dates, unemp_values, color="green", linewidth=1.5)
ax3.set_ylabel("Unemployment Rate (%)")
ax3.set_xlabel("Date")
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

## 5. Workflow 1: Prompt Chaining

The NewsProcessingChain processes raw news articles through five sequential stages.
Each stage is a separate LLM call with the output passed as input to the next.

1. **Ingest** -- retrieve raw articles from NewsAPI via `tools/news_data.fetch_news()`
2. **Preprocess** -- convert articles into clean structured text
3. **Classify** -- filter for relevance to the target ticker
4. **Extract** -- identify investment-relevant facts and developments
5. **Summarize** -- produce a concise factual narrative

In [ ]:
# ── Workflow 1: Prompt Chaining ───────────────────────────────────────────────
# NewsProcessingChain runs five sequential LLM calls, each building on the
# previous stage's output. The chain is self-contained; news retrieval
# occurs inside ingest() via tools/news_data.fetch_news().
# See workflows/prompt_chain.py for full implementation.

chain = NewsProcessingChain(TICKER)

print("Stage 1: Ingest")
articles = chain.ingest()
print(f"  Retrieved {len(articles)} articles")
print()

print("Stage 2: Preprocess")
preprocessed = chain.preprocess(articles)
print(f"  Preprocessed text length: {len(preprocessed)} characters")
print()

print("Stage 3: Classify")
classified = chain.classify(preprocessed)
print(f"  Classified text length: {len(classified)} characters")
print()

print("Stage 4: Extract")
extracted = chain.extract(classified)
print(f"  Extracted text length: {len(extracted)} characters")
print()

print("Stage 5: Summarize")
summarized = chain.summarize(extracted)
print(f"  Summary length: {len(summarized)} characters")
print()
print("Chain output:")
print(summarized)

---

## 6. Workflow 2: Routing and Specialist Agents

The ContentRouter classifies incoming financial content by type and dispatches
to the appropriate specialist agent. Each specialist fetches its own data and
produces a narrative analysis. Routing is based on content type, not data source.

Note: Workflow 1 (Prompt Chaining) runs inside the NewsAnalyzer section below.
Ken Lai's full five-stage chain implementation will replace the current stub.

### Specialist Agent 1: EarningsAnalyzer

Analyzes quarterly earnings and income statement data. Dispatched by the router
when content is classified as EARNINGS.

In [ ]:
# ── EarningsAnalyzer ──────────────────────────────────────────────────────────
# Fetches earnings and income data via Alpha Vantage and produces narrative
# analysis covering trends, revenue growth, margins, and earnings surprises.
# Populates analysis["earnings_analysis"] in the shared analysis dict.

earnings_agent = EarningsAnalyzer(TICKER)
analysis = earnings_agent.run(analysis)

if analysis["earnings_analysis"]:
    print(f"Earnings analysis length: {len(analysis['earnings_analysis'])} characters")
    print()
    print(analysis["earnings_analysis"])
else:
    print(
        "Earnings analysis returned empty. Possible causes: API rate limit, "
        "network error, or invalid ticker. Check Alpha Vantage quota at "
        "alphavantage.co/support."
    )

In [ ]:
# ── Visualization: Earnings Surprise ─────────────────────────────────────────
# Shows how much reported earnings per share (EPS) differed from analyst estimates
# each quarter, as a percentage. Positive values mean reported EPS exceeded estimates.
# Sourced from Alpha Vantage via tools/fundamentals.py.
# Feeds EarningsAnalyzer for trend analysis.

dates = [q["fiscalDateEnding"] for q in reversed(earnings)]
surprises = [float(q["surprisePercentage"]) for q in reversed(earnings)]
colors = ["green" if s > 0 else "red" for s in surprises]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(dates, surprises, color=colors, alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_title(f"{TICKER} Earnings Surprise % by Quarter (most recent 8)")
ax.set_ylabel("Surprise %")
ax.set_xlabel("Fiscal Quarter End")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Earnings surprise data fed to EarningsAnalyzer for earnings trend analysis.

### Specialist Agent 2: MarketAnalyzer

Analyzes price trends, momentum, and macroeconomic context. Dispatched by the
router when content is classified as MARKET or MACRO.

In [ ]:
# ── MarketAnalyzer ────────────────────────────────────────────────────────────
# Fetches price history via yfinance and macro indicators via FRED API.
# Produces narrative analysis covering price trends, momentum, and how the
# macro environment relates to the stock.
# Populates analysis["market_analysis"] in the shared analysis dict.

market_agent = MarketAnalyzer(TICKER)
analysis = market_agent.run(analysis)

print(f"Market analysis length: {len(analysis['market_analysis'])} characters")
print()
print(analysis["market_analysis"])

### Specialist Agent 3: NewsAnalyzer

Processes recent financial news through the five-stage prompt chain and produces
a narrative analysis covering material developments, sentiment, implications,
and key uncertainties. Dispatched by the router when content is classified as NEWS.

In [ ]:
# ── NewsAnalyzer ──────────────────────────────────────────────────────────────
# Fetches news via NewsAPI, processes through all five chain stages, then
# produces narrative analysis covering material developments, sentiment,
# and implications for the stock.
# Populates analysis["news_analysis"] in the shared analysis dict.
# See workflows/prompt_chain.py for the full five-stage chain implementation.

chain_output = chain.run()
news_agent = NewsAnalyzer(TICKER)
analysis = news_agent.run(analysis, processed_news=chain_output)

print(f"News analysis length: {len(analysis['news_analysis'])} characters")
print()
print(analysis["news_analysis"])

### Routing Demonstration

The ContentRouter uses an LLM call at temperature 0.2 to classify content
and dispatch to the correct specialist. The following demonstrates classification
across all three content types.

In [ ]:
# ── Workflow 2: Routing Demonstration ────────────────────────────────────────
# ContentRouter classifies content into NEWS, EARNINGS, MARKET, or MACRO
# and dispatches to the matching specialist agent.
# Temperature 0.2 used for classification; deterministic output preferred.
# See workflows/router.py and agents/schema.py (ContentType enum).

router = ContentRouter(TICKER)

test_inputs = [
    "Apple reported EPS of 2.02, beating estimates of 1.88",
    "Apple faces a 2.7 billion lawsuit over App Store tracking rules",
    "Federal Reserve holds interest rates at 3.63 percent amid inflation concerns",
]

print("Router classification test:")
print()
for content in test_inputs:
    ct = router.classify(content)
    specialist = router.specialists[ct].__class__.__name__
    print(f"  Input:      {content[:60]}...")
    print(f"  Classified: {ct.value} -> {specialist}")
    print()

---

## 7. Workflow 3: Evaluator-Optimizer

*Placeholder: Ken Lai's implementation pending.*

The EvaluatorOptimizer runs a generate-evaluate-refine loop on the synthesized
analysis. Each iteration scores the output on a 0-10 rubric and provides
feedback. The loop terminates when the score meets the threshold or maximum
iterations are reached.

Score range 0-10 was chosen over 0-100 based on research showing 1-100 scales
produce round-number bias and range underutilization in LLM evaluators
(Doosterlinck et al., 2024; EvidentlyAI, 2024).

---

## 8. Agent Function: Planning

The InvestmentResearchAgent generates a multi-step research plan for the given
ticker before any data is retrieved. Prior memory is read at this stage so the
planner can address gaps identified in previous analyses.

In [ ]:
# Section 8 -- Agent Function: Planning
# Demonstrate the planning stage in isolation, showing how prior memory
# informs the research agenda for the current run.

from agents.investment_agent import InvestmentResearchAgent
from memory.agent_memory import load_memory

agent = InvestmentResearchAgent(TICKER)

# Show prior memory context the planner will reference
prior = load_memory(TICKER)
if prior:
    print(f"Prior memory found for {TICKER} (score: {prior['evaluation_score']}/10)")
    print(f"Planner will address gaps from: {prior['timestamp']}\n")
else:
    print(f"No prior memory for {TICKER} -- planner will generate baseline plan\n")

# Generate the research plan
plan = agent.plan(prior_memory=prior)
print("=== Research Plan ===")
print(plan)

---

## 9. Agent Function: Self-Reflection

After synthesis and evaluation, the InvestmentResearchAgent reflects on the
quality of its own output, scoring completeness, factual grounding, and clarity.
The reflection narrative is persisted to memory for use in future runs.

In [ ]:
# Section 9 -- Agent Function: Self-Reflection
# Demonstrate the reflection stage in isolation using the saved analysis
# from the most recent pipeline run.

from agents.investment_agent import InvestmentResearchAgent
from memory.agent_memory import load_memory

agent = InvestmentResearchAgent(TICKER)

# Load the most recent analysis from memory to reflect on
prior = load_memory(TICKER)
if not prior:
    print(f"No prior analysis found for {TICKER} -- run Section 11 first.")
else:
    reflection = agent.reflect(prior)
    print("=== Self-Reflection ===")
    print(reflection)

---

## 10. Agent Function: Memory

The AgentMemory module persists completed analysis dicts to
`data/agent_memory.json`, keyed by ticker and capped at 5 entries per ticker.
On subsequent runs, the planner reads the most recent entry to inform its
research strategy and address prior gaps.

In [ ]:
# Section 10 -- Agent Function: Memory
# Demonstrate the memory module in isolation: inspect the persisted entry
# for the current ticker and show the planner will use on the next run.

from memory.agent_memory import load_memory

# Load most recent memory entry for this ticker
entry = load_memory(TICKER)

if entry:
    print(f"Most recent memory entry for {TICKER}:")
    print(f"  Timestamp:        {entry['timestamp']}")
    print(f"  Evaluation score: {entry['evaluation_score']}/10")
    print(f"  Synthesis length: {len(entry['synthesis'])} characters")
    print(f"  Reflection saved: {len(entry['reflection'])} characters")
    print(f"\nThis entry will be passed to the planner on the next run")
    print(f"to address gaps identified in the reflection.")
else:
    print(f"No memory found for {TICKER} -- run Section 11 first.")

---

## 11. Full Pipeline: InvestmentResearchAgent

The InvestmentResearchAgent orchestrates the complete research pipeline in a
single `agent.run()` call. To demonstrate Agent Function: Memory, the pipeline
is run twice. Run 1 establishes a baseline with no prior context. Run 2 reads
the memory and reflection saved by Run 1, allowing the planner to address
identified gaps.

Note: This notebook uses live API data. Re-execution will produce different
outputs as market data, news, and macro indicators change over time. The
memory file (`data/agent_memory.json`) is gitignored and not included in
the repository; it is created at runtime and persists between runs within
a local environment.

```
Ticker Input
    |
Planning (reads memory)
    |
Dynamic Tool Use (4 APIs)
    |
Prompt Chain (Workflow 1)
    |
Routing (Workflow 2)
    |
Specialist Agents (Earnings, News, Market)
    |
Synthesis
    |
Evaluator-Optimizer (Workflow 3)
    |
Self-Reflection
    |
Memory Write
    |
Final Investment Analysis Report
```

### Run 1: Baseline Analysis

No prior memory exists for this ticker. Run 1 establishes a baseline
analysis and saves the results to memory for Run 2 to build on.

In [ ]:
# Clear memory to guarantee Run 1 starts with no prior context
clear_memory(TICKER)

# Run 1 -- no prior memory
agent = InvestmentResearchAgent(TICKER)
analysis_run1 = agent.run()

print()
print("=== Run 1 Results ===")
print(f"Evaluation score: {analysis_run1['evaluation_score']}/10")
print(f"Synthesis length: {len(analysis_run1['synthesis'])} characters")
print()
print(analysis_run1["synthesis"])

### Agent Function: Memory

Run 1 is complete. The analysis has been saved to `data/agent_memory.json`,
keyed by ticker. The memory entry includes the synthesis, evaluation score,
reflection, and timestamp. Run 2's planner will read this entry and incorporate
the prior reflection to address identified gaps.

In [ ]:
# Show memory saved by Run 1
prior = load_memory(TICKER)
if prior:
    print(f"Memory entry for {TICKER}:")
    print(f"  Timestamp:        {prior['timestamp']}")
    print(f"  Evaluation score: {prior['evaluation_score']}/10")
    print(f"  Synthesis length: {len(prior['synthesis'])} characters")
    print(f"  Reflection saved: {len(prior['reflection'])} characters")
else:
    print(f"No memory found for {TICKER}.")

### Run 2: Memory-Informed Analysis

The planner reads the memory saved by Run 1 and incorporates the prior
reflection into its research strategy. An improvement in synthesis quality
or evaluation score between runs demonstrates Agent Function: Memory.

In [ ]:
# Run 2 -- planner reads Run 1 memory
analysis_run2 = agent.run()

print()
print("=== Run 2 Results ===")
print(f"Evaluation score: {analysis_run2['evaluation_score']}/10")
print(f"Synthesis length: {len(analysis_run2['synthesis'])} characters")
print()
print(analysis_run2["synthesis"])
print()
print("=== Score Comparison ===")
print(f"Run 1: {analysis_run1['evaluation_score']}/10")
print(f"Run 2: {analysis_run2['evaluation_score']}/10")

---

### References

Doosterlinck, K., et al. (2024). Evaluating the consistency of LLM evaluators.
https://arxiv.org/html/2412.00543v1

EvidentlyAI. (2024). LLM-as-a-judge guide.
https://www.evidentlyai.com/llm-evaluation/llm-as-a-judge

Mokhtari Jadid, K. (2026, September). [Response to student question on RAG
requirement]. AAI-520 course Slack channel, University of San Diego.